Before running this notbook, please run codonaligner.ipynb and filting_introns.ipynb

In [1]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm
import os

import trinuc_models as trinucs # this module must be in the same directory as this notebook

## CDS

In [2]:
#Setting up the rules for model fitting of cds regions
sm_noncds=trinucs.GNC_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_cds = [{"par_name": n, "is_independent": True} for n in paramnames]
GNC_subsmodel = get_app("model", "GNC_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_cds)

#Setting up the app for reading cds sequences
loader_cds = get_app("load_aligned", moltype="dna")   
omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

#create a concatenated alignment with all coding positions
cds_process = loader_cds + omit_degs_cds

In [3]:

folder_in = paths.DATA_HUMCHIMPGOR115 + 'cds/chrm22/codon_aligned'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_cds = [r for r in cds_process.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

cds_alns = concat(nonconcat_cds)
cds_alns.source = "cds_alignments"

result_cds = GNC_subsmodel(cds_alns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_cds.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_cds))

result_cds.lf

   0%|          |00:00<?

/home/u12/uliseshmc/.conda/envs/UdChimpHumOran/lib/python3.13/site-packages/cogent3/recalculation/definition.py:690: UserWarning: using slow exponentiator because 'eigen failed precision test'
  return func(*args)


   0%|          |00:00<?

GNC_CpG_ss
log-likelihood = -777109.3849
number of free parameters = 102
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        0.02               3.59    0.97    1.93    0.83
Gorilla       root        0.03               5.06    1.19    2.95    0.87
Human         root        0.01               4.46    1.13    3.67    0.68
-------------------------------------------------------------------------

continued: 
=====================================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C    omega
---------------------------------------------------------------------
0.79    1.12    2.17    1.87    1.33    0.94    0.64    1.94     0.45
1.00    1.47    2.90    2.45    1.55    1.30    0.68    2.94     0.33
1.22    1.56    4.45    3.66    1.36    1.15    0.83    2.84     0.35
---------------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.02    0.04    0.01    0.01    0.02    0.01    0.01    0.01    0.03
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.00    0.02    0.02    0.01    0.01    0.02    0.04    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.01    0.02    0.01    0.02    0.02    0.00    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.05    0.01    0.02    0.03    0.05    0.02    0.01    0.04    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAC     TAT
----------------------------------------------------------------------------
0.01    0.03    0.02    0.01    0.00    0.02    0.03    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TCA     TCC     TCG     TCT     TGC     TGG     TGT     TTA     TTC     TTG
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
====
 TTT
----
0.01
----

# Non cds

In [4]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_noncds)

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Orangutan, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [5]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'intergenicAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGAR = concat(nonconcat_IGAR)
alns_IGAR.source = "IGAR_alignments"

result_IGAR = GT_subsmodel(alns_IGAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_IGAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR))

result_IGAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -63677.5264
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        1.22              38.74    0.60    0.49    0.07
Gorilla       root        2.23              14.52    0.67    0.72    0.65
Human         root        1.01              50.00    0.33    0.40    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.43    0.29    0.16    0.09    0.24    0.10    1.13    0.99
0.38    0.52    0.50    0.34    0.60    0.40    1.17    0.92
0.07    0.19    0.10    0.06    0.12    0.00    1.10    0.95
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.00    0.01    0.02    0.01    0.01    0.00    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.01    0.02    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.02    0.01    0.03    0.00    0.00    0.01    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.02    0.01    0.01    0.02    0.01    0.01    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.02    0.01    0.00    0.01    0.01    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.02    0.01    0.02    0.01    0.03    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.02    0.02    0.01    0.23
----------------------------

## Introns

In [6]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'introns/chrm22/noUTRs/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_introns = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_introns = concat(nonconcat_introns)
alns_introns.source = "introns_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_introns = GT_subsmodel(alns_introns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_introns.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_introns))

result_introns.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -122765.8527
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        0.95              26.06    2.29    2.04    2.46
Gorilla       root        2.32              14.45    1.55    1.52    1.78
Human         root        0.83              42.63    2.73    2.39    2.44
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.39    0.82    0.49    0.53    1.25    1.12    0.57    1.10
0.76    1.15    0.65    0.88    1.18    0.91    1.16    1.13
0.04    0.58    0.30    0.38    1.26    0.74    0.00    1.28
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.08    0.01    0.04    0.02    0.01    0.01    0.02    0.01    0.04    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.01    0.01    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.01    0.02    0.01    0.02    0.01    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.01    0.03    0.01    0.02    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.02    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.02
----------------------------

## 5' UTR

In [7]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'introns/chrm22/5UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_5UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_5UTR = concat(nonconcat_5UTR)
alns_5UTR.source = "5UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_5UTR = GT_subsmodel(alns_5UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_5UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_5UTR))

result_5UTR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -81306.6585
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        0.54              10.46    0.82    0.79    0.69
Gorilla       root        1.32               7.47    0.78    0.89    0.85
Human         root        0.51              14.18    0.71    0.74    0.51
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.83    0.93    0.73    0.70    0.85    0.94    1.14    0.84
0.70    0.97    0.65    0.72    0.99    0.79    0.98    0.84
0.68    0.66    0.49    0.46    0.86    0.49    0.74    1.02
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.02    0.03    0.00    0.01    0.02    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.02    0.01    0.02    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.02    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.02    0.03
----------------------------

## 3' UTR

In [8]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'introns/chrm22/3UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_3UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_3UTR = concat(nonconcat_3UTR)
alns_3UTR.source = "3UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_3UTR = GT_subsmodel(alns_3UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_3UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_3UTR))

result_3UTR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -57107.8523
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Chimpanzee    root        1.85              50.00    38.83    44.31    44.67
Gorilla       root        3.01              19.92     2.45     2.58     3.29
Human         root        2.08              50.00    27.43    27.16    21.13
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.20    5.13    0.54    0.00    0.00    0.94
0.85    1.28    0.24    0.95    1.11    0.43    0.85    1.37
0.00    0.01    0.18    2.67    0.55    0.00    0.00    0.00
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.39    0.01    0.04    0.01    0.00    0.01    0.02    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.01    0.01    0.00    0.01    0.01    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.02    0.01    0.01    0.02    0.02    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.02    0.00    0.00    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.00    0.01    0.01    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.01    0.00    0.01
----------------------------

## Introns Ancestral Repeats (IntronAR)

In [9]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'intronsAR/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_intronsAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_intronsAR = concat(nonconcat_intronsAR)
alns_intronsAR.source = "intronsAR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_intronsAR = GT_subsmodel(alns_intronsAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_intronsAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_intronsAR))

result_intronsAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -110574.2652
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Chimpanzee    root        2.47              50.00    20.86    21.10    22.46
Gorilla       root        2.48              50.00     8.15     8.36     9.03
Human         root        2.44              50.00    49.02    48.76    42.41
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.12    2.09    0.26    2.39    1.38    0.39    0.34    2.48
0.21    0.97    0.08    1.02    0.77    0.52    0.15    1.00
0.00    0.00    0.33    4.65    0.00    0.00    0.00    0.27
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.54    0.01    0.03    0.01    0.00    0.01    0.01    0.01    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.01    0.01    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.01    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.01    0.00    0.01    0.01    0.02    0.00    0.00    0.01    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.00    0.00    0.01    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.01    0.00    0.01    0.01    0.01    0.01    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.00    0.00    0.01
----------------------------

## Intergenic distal (5kb distal from transcript)

In [10]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'distalIG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_distalIG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_distalIG = concat(nonconcat_distalIG)
alns_distalIG.source = "distalIG_alignments"

result_distalIG = GT_subsmodel(alns_distalIG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_distalIG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_distalIG))

result_distalIG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -22491.6885
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        0.48              10.10    1.11    1.19    0.62
Gorilla       root        0.54              13.78    0.95    1.04    0.77
Human         root        0.88              10.11    1.08    1.14    0.82
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.39    0.96    0.81    1.09    0.70    0.65    0.80    1.79
0.55    0.87    0.43    0.65    0.69    0.72    0.69    1.09
0.79    0.89    0.71    0.72    1.05    0.69    1.31    1.07
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.01    0.02    0.02    0.02    0.01    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.01    0.02    0.01    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.03    0.03    0.01    0.02    0.01    0.00    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.02    0.02    0.02    0.02    0.01    0.02    0.02    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.02    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.01    0.02    0.01    0.01    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.02    0.02
----------------------------

## Intergenic proximal 5' (5kb proximal from transcript start)

In [11]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'proximal5IG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal5IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal5IG = concat(nonconcat_proximal5IG)
alns_proximal5IG.source = "proximal5IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal5IG = GT_subsmodel(alns_proximal5IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal5IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal5IG))

result_proximal5IG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -23780.3576
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Chimpanzee    root        0.71               9.91    0.14    0.56    0.00
Gorilla       root        2.97              15.43    1.10    0.96    0.51
Human         root        0.61              12.99    0.00    0.24    0.00
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.73    0.97    0.59    0.34    0.00    0.14    0.79    0.52
0.61    1.17    0.69    0.44    0.93    0.47    1.10    0.97
0.59    0.93    0.46    0.14    0.00    0.00    0.81    0.19
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.00    0.01    0.01    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.00    0.01    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.08    0.02    0.06    0.00    0.01    0.01    0.01    0.00    0.04
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.03    0.01    0.01    0.02    0.01    0.02    0.02    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.02    0.03    0.02    0.00    0.01    0.02    0.01    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.01    0.02    0.04    0.00    0.04    0.01    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.00    0.03    0.01    0.03
----------------------------

## Intergenic proximal 3' (5kb proximal from transcript end)

In [12]:
folder_in = paths.DATA_HUMCHIMPGOR115 + 'proximal3IG/chrm22/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal3IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=False) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal3IG = concat(nonconcat_proximal3IG)
alns_proximal3IG.source = "proximal3IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal3IG = GT_subsmodel(alns_proximal3IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

os.makedirs(folder_in + "/fitted_models", exist_ok=True)

with open(folder_in + "/fitted_models/likelihood_proximal3IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal3IG))

result_proximal3IG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -25760.9225
number of free parameters = 102
=====
omega
-----
 1.00
-----
============================================================================
edge          parent    length    (CG>TG | CG>CA)      A>C      A>G      A>T
----------------------------------------------------------------------------
Chimpanzee    root        2.01              50.00    30.71    27.70    27.35
Gorilla       root        3.23              24.92     3.67     3.09     3.23
Human         root        2.01              50.00    46.17    50.00    40.89
----------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.00    0.00    0.20    3.52    0.00    0.00    0.00    0.95
1.01    0.98    0.24    0.87    0.58    0.82    1.30    1.11
0.00    0.00    0.11    5.85    0.00    0.00    0.00    0.00
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.44    0.01    0.01    0.01    0.00    0.01    0.02    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.01    0.02    0.00    0.01    0.01    0.00    0.00    0.00    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.00    0.01    0.02    0.01    0.01    0.01    0.02    0.01    0.00    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.02    0.01    0.01    0.01    0.02    0.00    0.00    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.01    0.01    0.01    0.01    0.00    0.01    0.00    0.00    0.01    0.00
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.00    0.00    0.00    0.01    0.01    0.01    0.01    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.00    0.01
----------------------------